In [ ]:
from google.colab import files
uploaded = files.upload()  # select clean_master_tract_1.csv from computer

In [ ]:
import pandas as pd

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv("clean_master_tract@1.csv")

# ── 1. Clean tract label ──────────────────────────────────────────────────────
# "Tract 1.02;PhiladelphiaCounty;Pennsylvania" → "Tract 1.02"
df["tract_label"] = df["tract.y"].str.split(";").str[0].str.strip()

# ── 2. Clean tract ID ─────────────────────────────────────────────────────────
# Drop the trailing .0 from the GEOID so it matches GeoJSON later
df["tract_id"] = df["tract"].astype(int).astype(str)

# ── 3. Assessment gap (absolute dollars) ─────────────────────────────────────
df["assessment_gap"] = df["median_assessed_value"] - df["median_sale_price"]

# ── 4. Over/under assessment flag ────────────────────────────────────────────
# ratio > 1 = over-assessed, < 1 = under-assessed
df["assessment_direction"] = df["median_ratio"].apply(
    lambda r: "Over-assessed" if r > 1 else "Under-assessed"
)

# ── 5. Drop columns we don't need for viz ────────────────────────────────────
df = df.drop(columns=["tract", "tract.y", "COD", "PRD"])

# ── 6. Split into two CSVs ───────────────────────────────────────────────────

# P1 — Barbell chart (2023 only, one row per tract)
barbell = df[df["year"] == 2023].copy()
barbell = barbell.sort_values("median_ratio")
barbell.to_csv("barbell_2023.csv", index=False)
print(f"barbell_2023.csv: {len(barbell)} rows")

# P2 — Temporal line chart (all years, aggregated by majority_race)
temporal = (
    df.groupby(["year", "majority_race"])
    .agg(
        median_sale_price=("median_sale_price", "median"),
        median_assessed_value=("median_assessed_value", "median"),
        median_ratio=("median_ratio", "median"),
        n_tracts=("tract_id", "count"),
    )
    .reset_index()
)
temporal.to_csv("temporal_by_race.csv", index=False)
print(f"temporal_by_race.csv: {len(temporal)} rows")

# Full cleaned file (for scatterplots / choropleths)
df.to_csv("master_clean.csv", index=False)
print(f"master_clean.csv: {len(df)} rows")

print("\nDone! Three files ready for Observable:")
print("  barbell_2023.csv   → P1 barbell + hex map")
print("  temporal_by_race.csv → P2 line chart")
print("  master_clean.csv   → P3 scatterplots / everything else")

barbell_2023.csv: 49 rows
temporal_by_race.csv: 29 rows
master_clean.csv: 411 rows

Done! Three files ready for Observable:
  barbell_2023.csv   → P1 barbell + hex map
  temporal_by_race.csv → P2 line chart
  master_clean.csv   → P3 scatterplots / everything else


In [ ]:
files.download("barbell_2023.csv")
files.download("temporal_by_race.csv")
files.download("master_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
df = pd.read_csv("master_clean.csv")
print(df['majority_race'].value_counts())
print(df[['tract_label', 'majority_race', 'pct_black', 'pct_white', 'pct_hispanic']].head(20))

FileNotFoundError: [Errno 2] No such file or directory: 'master_clean.csv'